In [1]:
%pip install numpy
%pip install matplotlib
%pip install torch

  Using cached numpy-2.4.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.4.4-cp312-cp312-win_amd64.whl (12.3 MB)
Note: you may need to restart the kernel to use updated packages.
  Using cached matplotlib-3.10.9-cp312-cp312-win_amd64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp312-cp312-win_amd64.whl.metadata (119 kB)
  Using cached kiwisolver-1.5.0-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.2.0-cp312-cp312-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.10.9-cp312-cp312-win_amd64.whl (8.2 MB)
Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.62.1-cp312-cp312-win_amd64.whl (2.3 MB)
Using cached kiwisolver-1.5.0-cp312-cp312-win_am

In [2]:
import os
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split

In [ ]:
data_base_dir = "/data"

In [ ]:


if not os.path.exists(data_base_dir):
    candidate = "./data/"
    if os.path.exists(candidate):
        data_base_dir = candidate
    else:
        data_base_dir = "./data/"

print("Using data dir:", data_base_dir)

Using data dir: ./data/


In [11]:
x_path = os.path.join(data_base_dir, "gaussian_noise.npy")
y_path = os.path.join(data_base_dir, "labels.npy")

X = np.load(x_path)     # expected shape: (50000, 32, 32, 3)
y = np.load(y_path)     # expected shape: (50000,)

print("X shape:", X.shape, "dtype:", X.dtype)
print("y shape:", y.shape, "dtype:", y.dtype)
print("num classes:", len(np.unique(y)))

X shape: (50000, 32, 32, 3) dtype: uint8
y shape: (50000,) dtype: uint8
num classes: 10


In [12]:
X_tensor = torch.tensor(X, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0  # (N, 3, 32, 32)
y_tensor = torch.tensor(y, dtype=torch.long)

In [13]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                             # 16x16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                             # 8x8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [15]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            total_loss += criterion(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

In [16]:
# ── CONFORMAL PREDICTION ──────────────────────────────────────────────────────
import math

ALPHA = 0.1

# 1. split: train / calibration / test
dataset = TensorDataset(X_tensor, y_tensor)
n = len(dataset)
train_size = int(0.6 * n)
cal_size   = int(0.2 * n)
test_size  = n - train_size - cal_size

train_ds, cal_ds, test_ds = random_split(dataset, [train_size, cal_size, test_size])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
cal_loader   = DataLoader(cal_ds,   batch_size=64)
test_loader  = DataLoader(test_ds,  batch_size=64)

In [17]:
# 2. (Re-)train model
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(10):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    val_loss, val_acc     = evaluate(model, cal_loader, criterion)
    print(f"Epoch {epoch+1:2d} | Train {train_loss:.4f}/{train_acc:.3f} | Cal {val_loss:.4f}/{val_acc:.3f}")

Epoch  1 | Train 1.6148/0.415 | Cal 1.2934/0.535
Epoch  2 | Train 1.2073/0.567 | Cal 1.0384/0.637
Epoch  3 | Train 0.9802/0.652 | Cal 0.8475/0.711
Epoch  4 | Train 0.7555/0.737 | Cal 0.6365/0.786
Epoch  5 | Train 0.5703/0.804 | Cal 0.5176/0.832
Epoch  6 | Train 0.4300/0.853 | Cal 0.3840/0.875
Epoch  7 | Train 0.3218/0.890 | Cal 0.2972/0.912
Epoch  8 | Train 0.2515/0.915 | Cal 0.2341/0.932
Epoch  9 | Train 0.2003/0.934 | Cal 0.1813/0.949
Epoch 10 | Train 0.1670/0.943 | Cal 0.1538/0.957


In [19]:
# 3. Calibration – collect nonconformity scores s_i = 1 - ŷ[true_class]
model.eval()
cal_scores = []
with torch.no_grad():
    for X_batch, y_batch in cal_loader:
        probs = torch.softmax(model(X_batch.to(device)), dim=1).cpu()
        true_class_probs = probs[torch.arange(len(y_batch)), y_batch]
        cal_scores.append(1.0 - true_class_probs)

cal_scores = torch.cat(cal_scores)          # shape: (n_cal,)

# Finite-sample corrected quantile level (Angelopoulos & Bates, 2022, Section 1.1, Step 3)
n_cal = len(cal_scores)
q_level = math.ceil((n_cal + 1) * (1 - ALPHA)) / n_cal
q_hat = torch.quantile(cal_scores, min(q_level, 1.0))
print(f"\nCalibration quantile q̂ = {q_hat:.4f}  (α={ALPHA}, n_cal={n_cal})")


Calibration quantile q̂ = 0.2494  (α=0.1, n_cal=10000)


In [20]:
# 4. Test – build prediction sets and measure coverage / efficiency
model.eval()
covered, total, set_sizes = 0, 0, []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        probs = torch.softmax(model(X_batch.to(device)), dim=1).cpu()

        pred_sets = probs >= (1.0 - q_hat)

        for i, y_true in enumerate(y_batch):
            in_set = pred_sets[i, y_true.item()].item()
            covered += int(in_set)
            set_sizes.append(pred_sets[i].sum().item())
        total += len(y_batch)

empirical_coverage = covered / total
avg_set_size       = sum(set_sizes) / len(set_sizes)
print(f"Empirical coverage : {empirical_coverage:.4f}  (target ≥ {1-ALPHA:.2f})")
print(f"Avg prediction-set size: {avg_set_size:.3f}  (out of 10 classes)")

Empirical coverage : 0.9011  (target ≥ 0.90)
Avg prediction-set size: 0.917  (out of 10 classes)


In [21]:
# 5. Per-class coverage
classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
print("\nPer-class coverage:")
class_covered = [0]*10;  class_total = [0]*10
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        probs = torch.softmax(model(X_batch.to(device)), dim=1).cpu()
        pred_sets = probs >= (1.0 - q_hat)
        for i, y_true in enumerate(y_batch):
            c = y_true.item()
            class_covered[c] += int(pred_sets[i, c].item())
            class_total[c]   += 1
for i, cls in enumerate(classes):
    print(f"  {cls:<12}: {class_covered[i]/class_total[i]:.3f}")


Per-class coverage:
  airplane    : 0.930
  automobile  : 0.954
  bird        : 0.824
  cat         : 0.854
  deer        : 0.858
  dog         : 0.855
  frog        : 0.905
  horse       : 0.934
  ship        : 0.949
  truck       : 0.947
